# PUBG: Khai phá hành vi người chơi trên Colab Pro

Notebook Colab riêng được yêu cầu ngày20/10/2026. Giữ ba RQ, toàn bộ dữ liệu hợp lệ, match split70/15/15, KMeans per_mode, OLS/baselines và Combat Timing của bản5.11. Không chứa kết quả PUBG hoặc checkpoint cũ. Code src/config/requirements được đóng kèm ngay trong notebook, không cần tải Dataset code riêng.

I. Bắt đầu

1. Chọn Runtime > Change runtime type > GPU, bật High RAM nếu tài khoản có lựa chọn này. Notebook dùng GPU0 được cấp thực tế; Pro không bảo đảm loại GPU, số GPU, RAM hoặc thời gian vô hạn.
2. Chạy cell1 cấu hình và cell2 cài môi trường. Nếu cell2 báo đã cài: chọn Runtime > Restart session, rồi Runtime > Run all. Không chọn Disconnect and delete runtime. Khi đủ phiên bản, cell2 tự bỏ qua cài lại.
3. Cho phép kết nối Drive khi được hỏi. Cell3 tải đúng Dataset Version3 từ Kaggle vào ổ runtime, không nạp tất cả CSV vào pandas. Public dataset thường không cần token; khi cần, thêm KAGGLE_API_TOKEN vào Colab Secrets và cấp quyền notebook, hoặc đặt KAGGLE_LOGIN=True. Không dán token vào code/output hay gửi cho người khác.

II. Lưu trữ

- Mặc định project/checkpoint/Parquet/model/release nằm trong thư mục Drive riêng PUBG_Coursework_Colab_v511. Raw/cache, SQL/spill và ma trận tạm nằm ở runtime. Không sao cả project sau mỗi stage. Drive có thể chậm khi đọc JOIN nhiều lượt; không hứa tăng tốc.
- Cần tự kiểm tra Drive còn tối thiểu khoảng25GiB; Colab Pro không phải gói tăng dung lượng Drive. Ngân sách project vẫn18GiB, không phải quota thực. Runtime còn ít nhất55GiB trước tải là dự phòng, không bảo đảm toàn pipeline vừa tài nguyên.
- Khi mất phiên: mở lại đúng notebook/root, cài lại môi trường nếu cần, tải raw đúngVersion3 rồi Run all. Chỉ reuse stage completed có code/config/software/source/checksum khớp; phiên bản phần mềm đổi sẽ bị chặn, không ép manifest/checksum. Stage đang chạy dở phải chạy lại, takeover chỉ sau khi xác nhận writer cũ đã dừng. Không chạy đồng thời hai phiên ghi cùng root.
- Nếu USE_GOOGLE_DRIVE=False thì toàn project chỉ ở ổ tạm và có thể mất khi VM bị thu hồi. Lưu file notebook KHÔNG đồng nghĩa lưu dữ liệu hoặc model.

III. Giới hạn

S01-S06/SQL và phần lớn I/O là CPU; S07/S08 mới dùng cuML GPU. GPU không cộng VRAM vào RAM. Full KMeans/OLS vẫn có thể vượt tài nguyên; không tự lấy mẫu hoặc đổi thuật toán. Giây vẫn là giả định được duyệt cho nguồnVersion3, không phải tác giả xác minh. Checksum đọc qua Drive mount không chứng minh đã đồng bộ lên máy chủ Google.

Nguồn vận hành: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Kagglehub](https://github.com/Kaggle/kagglehub), [RAPIDS installation](https://docs.nvidia.com/datascience/install/).


<a id="S00"></a>
## S00. Colab: môi trường, Kaggle Version3 và nơi lưu

Chạy cell1 cấu hình, cell2 môi trường; restart session một lần nếu được yêu cầu, rồi Run all. Cell3 kết nối Drive/tải raw, cell4 khởi tạo kiểm nguồn/đơn vị/GPU. Không mang checkpoint Kaggle sang Colab để ép reuse; đây là root mới với một GPU và software signature riêng.

In [ ]:
#@title 1. Cấu hình Colab Pro
EXECUTION_MODE = "full"  # full, development hoặc report_only; development KHÔNG tự lấy mẫu.
ALLOW_DATA_PROCESSING = True
ALLOW_FULL_DATA = True
USE_GOOGLE_DRIVE = True
PUBG_PROJECT_ROOT = "/content/drive/MyDrive/PUBG_Coursework_Colab_v511" if USE_GOOGLE_DRIVE else "/content/PUBG_Coursework_Colab_v511"
CODE_ROOT = "/content/pubg-code-511-4e3f9a19"
KAGGLE_CACHE_ROOT = "/content/pubg-kaggle-cache"
SOURCE_DATASET = "skihikingkevin/pubg-match-deaths"
SOURCE_VERSION = 3
GPU_IDS = [0]  # Một GPU Colab; không yêu cầu hai T4, không fallback CPU.
CUDA_FAMILY = "cu12"  # Phải khớp nvcc; nếu toolkit CUDA13 thì sửa thành cu13.
INSTALL_COLAB_DEPENDENCIES = True
KAGGLE_LOGIN = False  # Chỉ True nếu Kaggle yêu cầu đăng nhập/đồng ý điều khoản.
MIN_LOCAL_FREE_GIB = 55  # Dự phòng raw/cache/giải nén/SQL, không phải quota Colab đã đo.
START_STAGE = "S01"
END_STAGE = "S10"
RUN_GPU_SMOKE = True
PUBG_WRITER_ID = ""
PUBG_FORCE_STAGE_TAKEOVER = False  # Chỉ bật khi chắc phiên cũ đã dừng.
RELEASE_MANIFEST = ""  # report_only: đường dẫn release.json cụ thể trên Drive.
VERIFY_HANDOFF = True  # Cuối lượt đọc lại checksum/Parquet qua mount; có thể mất thời gian.


In [ ]:
#@title 2. Cài môi trường và mở gói code đi kèm
"""Colab transport/setup only; scientific code remains the frozen src bundle."""
import base64
import copy
import hashlib
import importlib.metadata
import io
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import subprocess
import sys
import tempfile
import zipfile


def unpack_code(payload, expected_hash, target):
    data = base64.b64decode(payload, validate=True)
    if hashlib.sha256(data).hexdigest() != expected_hash:
        raise ValueError("Embedded code checksum mismatch; use the original notebook")
    target = Path(target).resolve()
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        names = archive.namelist()
        expected = {"configs/project.yaml", "requirements.txt", "PUBG_RESEARCH_SPEC.md"}
        expected.update(n for n in names if re.fullmatch(r"src/[a-z_]+\.py", n))
        if len(names) != len(set(names)) or set(names) != expected or "src/io.py" not in names:
            raise ValueError("Unexpected embedded code paths")
        if sum(i.file_size for i in archive.infolist()) > 4 * 1024**2:
            raise ValueError("Embedded code is too large")
        for name in names:
            path = target.joinpath(*PurePosixPath(name).parts)
            if not path.resolve().is_relative_to(target):
                raise ValueError("Code path escapes target")
            if path.exists() and (not path.is_file() or path.read_bytes() != archive.read(name)):
                raise RuntimeError("Code folder contains different files; choose a new CODE_ROOT")
        # Validate every existing destination before writing anything.
        for name in names:
            path = target / name
            if not path.exists():
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(archive.read(name))
    return target


def environment_requirements(code_root, cuda_family):
    from packaging.requirements import Requirement
    if cuda_family not in {"cu12", "cu13"}:
        raise ValueError("CUDA_FAMILY must be cu12 or cu13, matching the installed CUDA toolkit")
    opposite = "cu13" if cuda_family == "cu12" else "cu12"
    for package in (f"cudf-{opposite}", f"cuml-{opposite}", f"cupy-cuda{opposite[2:]}x"):
        try:
            importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            continue
        raise RuntimeError("Conflicting CUDA family installed: " + package)
    requirements = [s.strip() for s in (Path(code_root) / "requirements.txt").read_text(encoding="utf-8").splitlines()
                    if s.strip() and not s.lstrip().startswith("#")]
    # 26.10 requires pandas3, outside the frozen core's pandas<3 contract.
    requirements += ["kagglehub==1.0.2", f"cuml-{cuda_family}==26.2.*", f"cudf-{cuda_family}==26.2.*",
                     f"cupy-cuda{cuda_family[2:]}x>=13.6,<14"]
    missing = []
    for line in requirements:
        requirement = Requirement(line)
        try:
            if importlib.metadata.version(requirement.name) not in requirement.specifier:
                missing.append(line)
        except importlib.metadata.PackageNotFoundError:
            missing.append(line)
    return requirements, missing


def setup_environment(code_root, cuda_family, install=True):
    if sys.platform != "linux":
        raise RuntimeError("This setup cell requires a Linux Colab GPU runtime")
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
    nvcc = shutil.which("nvcc")
    if nvcc:
        toolkit = subprocess.check_output([nvcc, "--version"], text=True)
        match = re.search(r"release (\d+)\.", toolkit)
        if match and "cu" + match[1] != cuda_family:
            raise RuntimeError("CUDA_FAMILY does not match nvcc; set it to cu" + match[1])
    requirements, missing = environment_requirements(code_root, cuda_family)
    if missing:
        if not install:
            raise RuntimeError("Enable INSTALL_COLAB_DEPENDENCIES; missing: " + ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements])
        raise RuntimeError("Đã cài môi trường. Chọn Runtime > Restart session, rồi Run all lại. Không Disconnect and delete runtime. Không tiếp tục S01 trong kernel cũ.")
    print("Môi trường đã có các phiên bản yêu cầu; không cài lại.")


def download_source(dataset, version, cache_root, minimum_free_gib=55, login=False):
    # Source-bound time-unit approval must not silently transfer to another dataset/version.
    if dataset != "skihikingkevin/pubg-match-deaths" or type(version) is not int or version != 3:
        raise ValueError("This notebook is approved for skihikingkevin/pubg-match-deaths/versions/3 only")
    cache_root = Path(cache_root).resolve()
    cache_root.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(cache_root).free / 2**30
    if free < minimum_free_gib:
        raise MemoryError(f"Local disk free {free:.1f} GiB < reserve {minimum_free_gib}; raw download, extraction and SQL need local disk")
    if "kagglehub" in sys.modules and os.environ.get("KAGGLEHUB_CACHE") != str(cache_root):
        raise RuntimeError("kagglehub was loaded with another cache; restart session")
    os.environ["KAGGLEHUB_CACHE"] = str(cache_root)
    import kagglehub
    if login:
        kagglehub.login()  # Interactive/secret authentication; never print or save tokens in this notebook.
    handle = f"{dataset}/versions/{version}"
    root = Path(kagglehub.dataset_download(handle)).resolve()
    groups = {"aggregate": sorted(root.rglob("agg_match_stats*.csv")),
              "event": sorted(root.rglob("kill_match_stats*.csv"))}
    if any(not paths for paths in groups.values()):
        raise FileNotFoundError("Pinned Kaggle download lacks aggregate or death CSVs")
    if any(not p.resolve().is_relative_to(root) for paths in groups.values() for p in paths):
        raise ValueError("Source link escapes download root")
    print("Kaggle handle:", handle, "; không dùng latest")
    print("Raw cache:", root)
    for name, paths in groups.items():
        print(name, ":", len(paths), "files;", round(sum(p.stat().st_size for p in paths) / 2**30, 3), "GiB")
    print("Chỉ kiểm tên/dung lượng; S01 mới kiểm đầy đủ hash/schema/rows.")
    return str(root)


def configure_project(code_root, target, source_directory, gpu_ids, scope):
    from src.io import Context, load_config, publish
    from src.kaggle_runtime import prepare_project
    import yaml
    if gpu_ids != [0] or any(type(i) is not int for i in gpu_ids):
        raise ValueError("Colab variant explicitly uses GPU_IDS=[0]; no silent CPU/second-GPU fallback")
    target = prepare_project(code_root, target)
    original = load_config(target)
    cfg = copy.deepcopy(original)
    cfg["source"].update(directory=source_directory, version=3, license="CC0: Public Domain")
    cfg["storage"]["project_root"] = str(target)
    cfg["runtime"]["temp_directory"] = None  # /tmp runtime, not the mounted Drive project.
    for section in ("rq2", "rq3"):
        cfg[section].update(device="cuda", gpu_ids=gpu_ids)
    # Check completed checkpoints against the candidate configuration before replacing a good config.
    check = Context(target, scope)
    check.cfg = cfg
    validated, hashes = set(), {}
    for stage, record in check.manifest().items():
        if record.get("status") == "completed":
            check.require(stage, validated, hashes)
    if cfg != original:
        with tempfile.TemporaryDirectory(prefix="pubg-colab-config-") as folder:
            path = Path(folder) / "project.yaml"
            path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
            publish(path, target / "configs/project.yaml")
    return target


def colab_handoff(ctx):
    from src.io import read_json, write_json
    from src.kaggle_runtime import prepare_handoff
    path = prepare_handoff(ctx)
    value = read_json(path)
    value.update(platform="Colab", saved_version_verified=False,
                 instructions="Keep this complete project on Drive, including src/config/data/outputs. Mounted-file read-back is not remote-cloud or Kaggle Version confirmation.")
    return write_json(value, path)


def verify_colab_handoff(root, scope):
    from src.io import digest, read_json
    from src.kaggle_runtime import safe_file
    from src.storage import validate_prediction
    if scope not in {"full", "development"}:
        raise ValueError("Invalid scope")
    root = Path(root).resolve()
    value = read_json(safe_file(root, f"handoff_{scope}.json"))
    if (value.get("format") != 1 or value.get("platform") != "Colab"
            or value.get("project_id") != "pubg-coursework-2026-v5" or value.get("scope") != scope):
        raise ValueError("Invalid Colab handoff marker")
    entries = value.get("entries", [])
    names = [entry["path"] for entry in entries]
    required = {"configs/project.yaml", "src/io.py", f"outputs/{scope}/checkpoints/stages.json"}
    if len(names) != len(set(names)) or not required.issubset(names) or list(root.glob(f"outputs/{scope}/checkpoints/*.lock")):
        raise ValueError("Incomplete/duplicate/locked Colab handoff")
    by_name = {entry["path"]: entry for entry in entries}
    for entry in entries:
        path = safe_file(root, entry["path"])
        if path.stat().st_size != entry["bytes"] or digest(path) != entry["sha256"]:
            raise ValueError("Colab handoff checksum mismatch: " + entry["path"])
        if path.suffix == ".parquet":
            validate_prediction(path)
    records = read_json(safe_file(root, f"outputs/{scope}/checkpoints/stages.json"))
    if records != value["stages"] or any(r.get("status") == "running" for r in records.values()):
        raise ValueError("Handoff stage records mismatch/running")
    for record in records.values():
        for entry in record.get("artifacts", []):
            if entry["path"] not in by_name or by_name[entry["path"]]["sha256"] != entry["sha256"]:
                raise ValueError("Missing/mismatched stage artifact")
    return {"readback_verified": True, "entries": len(entries), "scope": scope,
            "environment": "mounted filesystem read-back; not remote-cloud confirmation",
            "kaggle_version_verified": False}

BUNDLED_CODE = '...'
BUNDLED_CODE_SHA256 = '5166eebae3b685ed607130ada0d4799c35116a42f1142226f12467e80d80bc4d'
if EXECUTION_MODE not in {"full", "development", "report_only"}:
    raise ValueError("execution mode không hợp lệ")
if EXECUTION_MODE != "report_only":
    if not ALLOW_DATA_PROCESSING or (EXECUTION_MODE == "full" and not ALLOW_FULL_DATA):
        raise ValueError("Chưa cho phép xử lý/full-data")
    unpack_code(BUNDLED_CODE, BUNDLED_CODE_SHA256, CODE_ROOT)
    setup_environment(CODE_ROOT, CUDA_FAMILY, INSTALL_COLAB_DEPENDENCIES)
else:
    print("report_only: không giải nén/cài đặt; dùng code đã có trong project.")


In [ ]:
#@title 3. Kết nối Drive và tải dữ liệu Kaggle đã pin
if EXECUTION_MODE not in {"full", "development", "report_only"}:
    raise ValueError("execution mode không hợp lệ")
project = Path(PUBG_PROJECT_ROOT).resolve()
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    project = Path(PUBG_PROJECT_ROOT).resolve()
    drive_root = Path("/content/drive/MyDrive").resolve()
    if not project.is_relative_to(drive_root) or project == drive_root:
        raise ValueError("Chọn thư mục project riêng bên trong MyDrive, không dùng MyDrive làm project")
elif project in {Path("/"), Path("/content"), Path("/content/drive")}:
    raise ValueError("Không dùng root runtime hoặc mount làm project")
if EXECUTION_MODE != "report_only":
    if not ALLOW_DATA_PROCESSING or (EXECUTION_MODE == "full" and not ALLOW_FULL_DATA):
        raise ValueError("Chưa cho phép xử lý/full-data cho nguồn này")
    for location in (Path(CODE_ROOT).resolve(), Path(KAGGLE_CACHE_ROOT).resolve()):
        if project.is_relative_to(location) or location.is_relative_to(project):
            raise ValueError("Project phải tách khỏi code/cache raw")
    # Check actual CUDA imports/count in a fresh child before downloading ~19GiB.
    subprocess.check_call([sys.executable, "-c", "import cupy,cuml; assert cupy.cuda.runtime.getDeviceCount() >= 1; print('cuML',cuml.__version__,'GPU count',cupy.cuda.runtime.getDeviceCount())"])
    SOURCE_DIRECTORY = download_source(SOURCE_DATASET, SOURCE_VERSION, KAGGLE_CACHE_ROOT, MIN_LOCAL_FREE_GIB, KAGGLE_LOGIN)
else:
    print("report_only: bỏ qua GPU/raw download; chỉ mount để đọc release.")


In [ ]:
import runpy
sys.dont_write_bytecode = True
if EXECUTION_MODE not in {"full", "development", "report_only"}:
    raise ValueError("execution mode phải full/development/report_only")
if START_STAGE not in [f"S{i:02d}" for i in range(1,11)] or END_STAGE not in [f"S{i:02d}" for i in range(1,11)] or START_STAGE > END_STAGE:
    raise ValueError("Stage range phải S01..S10 và START<=END")
project_root = Path(PUBG_PROJECT_ROOT).resolve()
import_root = project_root if EXECUTION_MODE == "report_only" else Path(CODE_ROOT).resolve()
bundle = runpy.run_path(str(import_root/"src/bootstrap.py"))["check_bundle"](import_root, "5.11-normalized-storage", "4e3f9a191e589c963727c12bbf00ec656ee6bf17de0621ca6a6f520b3641dd3c")
print(bundle)
sys.path.insert(0, str(import_root))
from IPython.display import display, Image
import pandas as pd
import yaml
from src.io import Context, PendingDecision, read_json, load_config, stage_options, time_unit_info
from src.kaggle_runtime import prepare_project, preflight, gpu_smoke, prepare_handoff, verify_handoff, restore_handoff, resource_plan, import_s01_510, cleanup_imported_510
from src.preprocessing import ingest, clean_split
from src.features import base_features, combat_timing
from src.analysis import eda, relationship, clustering
from src.modeling import prediction, comparisons
from src.reporting import finalize, display_release
if EXECUTION_MODE != "report_only":
    project_root = configure_project(import_root, project_root, SOURCE_DIRECTORY, GPU_IDS, EXECUTION_MODE)
    progress_view = display(pd.DataFrame([{"step":"Sẵn sàng; chưa scan nguồn"}]), display_id=True)
    def show_progress(event):
        progress_view.update(pd.DataFrame([event]))
    ctx = Context(project_root, EXECUTION_MODE, PUBG_WRITER_ID, PUBG_FORCE_STAGE_TAKEOVER, progress=show_progress)
    display(ctx.environment())
    display(pd.DataFrame([preflight(ctx, START_STAGE, END_STAGE, kaggle=True)]))
    display(pd.DataFrame([time_unit_info(ctx.cfg)]))
    print("Project:", ctx.root, "; SQL/scratch runtime:", ctx.temp)
    print("Drive quota phải tự kiểm tra; disk runtime không phải quota Drive.")
    if RUN_GPU_SMOKE:
        display(pd.DataFrame([gpu_smoke(ctx)]))
else:
    load_config(project_root)
    if not RELEASE_MANIFEST:
        raise ValueError("report_only cần RELEASE_MANIFEST cụ thể, không lấy latest")
    print("Chỉ đọc release; không tải raw/cài đặt/build/train/ghi project.")


Cách đọc: version tải cố định3, unit assumed_seconds không phải verified. GPU smoke48dòng chỉ kiểm backend/reload. Các guard RAM/VRAM/đĩa/output và G4 vẫn giữ. Checkpoint ghi ngay trong project Drive; không cần Save Version Kaggle. Drive mount không bảo đảm quota hoặc đồng bộ máy chủ. Không xóa/ép lock nếu writer cũ còn chạy.

In [ ]:
STAGES = {"S01":ingest,"S02":clean_split,"S03":base_features,"S04":combat_timing,
          "S05":eda,"S06":relationship,"S07":clustering,"S08":prediction,"S09":comparisons,"S10":finalize}
def show_stage(stage):
    record = ctx.manifest().get(stage,{})
    paths = [ctx.root/a["path"] for a in record.get("artifacts",[])]
    if record.get("status") != "completed":
        paths = list(ctx.created)
    for path in paths:
        if path.suffix==".csv" and "assignments" not in path.name:
            print(path.name); display(pd.read_csv(path, nrows=20))
        elif path.suffix==".png":
            display(Image(filename=str(path)))
    display(pd.DataFrame([{"stage":stage,"status":record.get("status"),"reason":record.get("reason",""),
                           "artifacts":len(record.get("artifacts",[])),"checkpoint":str(ctx.manifest_path)}]))
def run_section(stage, options=None):
    if EXECUTION_MODE=="report_only":return
    if not START_STAGE <= stage <= END_STAGE:
        print(stage+": ngoài khoảng thực thi; không tính lại");return
    stage_options(ctx, stage, options)
    preflight(ctx, stage, stage, kaggle=True)
    if stage == 'S05' and END_STAGE >= 'S07':
        plan = resource_plan(ctx, stage, END_STAGE)
        display(pd.DataFrame(plan['checks']))
        print('RAM budget GiB:', plan['host_budget_GiB'], ';', plan['limitation'])
        if plan['feasible'] is False:
            raise MemoryError('Dữ liệu hợp lệ vượt RAM dự toán S07/S08; cần máy nhiều RAM hơn. Không giảm dữ liệu hoặc đổi thuật toán.')
    try:
        ctx.run(stage,STAGES[stage])
    finally:
        show_stage(stage)
        if stage == END_STAGE:
            try:
                print("Manifest Colab:", colab_handoff(ctx))
            except (RuntimeError, ValueError, MemoryError, OSError) as error:
                print("Chưa tạo được handoff; checkpoint stage vẫn cần kiểm:", str(error))
        else:
            print("Checkpoint đã ghi ở", ctx.manifest_path, "; chưa chứng minh đồng bộ Drive máy chủ.")


<a id="S01"></a>
## S01. Nguồn, schema và đối soát

Câu hỏi kiểm tra: Rows đọc và rows Parquet phải khớp; thiếu required hoặc alias collision dừng trước downstream. Metadata source/version/license/download_date chưa biết giữ unknown, không dùng mtime.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Một nhóm aggregate có player-match; một nhóm event có killer/victim/time. Discovery mọi shard, không hard-code hai file. Header required/alias phải đúng; parse errors khác missing gốc. CSV được đọc theo luồng, Parquet từng shard có lineage để phát hiện conflict xuyên shard. Các shard Parquet chỉ là file tạm: sau khi aggregate.parquet hoặc event.parquet đã được giải mã, kiểm số dòng và publish/read-back thành công, chúng được dọn khỏi thư mục tạm riêng của lượt chạy. Không giữ thư mục typed trùng dữ liệu, không bỏ raw columns hoặc lấy mẫu.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S01_OPTIONS = {
    'directory': None,
    'version': None,
    'license': None
}
run_section("S01", S01_OPTIONS)

Cách đọc kết quả: Inventory bar theo shard và parse/missing cho thấy dữ liệu đi vào; top20 chỉ preview, CSV nguồn đủ hàng. Raw mới giữ bất biến. source.directory null phải nhập sau xác định nguồn.

Hạn chế và bàn giao: Inventory/hash/schema/parse trong outputs; aggregate.parquet và event.parquet trong data cho consumer S02/S04. ingest_storage.csv ghi số dòng, số shard, byte file tạm đã dọn và byte bản gộp; temporary_files_remaining phải bằng 0 sau thành công. Đây không phải số đo đỉnh dung lượng toàn quy trình. Đơn vị chưa xác minh sẽ chặn S04.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S02"></a>
## S02. Chất lượng, roster và split
Bản 5.6: bảng trung gian ở database tạm trên đĩa; thông tin đội, roster, ngày và chế độ được tổng hợp riêng trước khi ghép. Không giảm dữ liệu hoặc tăng giới hạn DuckDB. s02_execution.json ghi cách thực thi.

Câu hỏi kiểm tra: Đối soát từng bước: before=removed+after. Một match chỉ một split; chronology Grade B không xé ngày, A không xé timestamp ties, C chia group match. Tỷ lệ70/15/15 thực đạt có thể lệch do nguyên block.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Exact duplicates được gộp xuyên shard theo nội dung nguồn; cùng identity khác nội dung giữ conflict audit và loại khỏi cohort theo quy tắc. Flags lỗi chồng lặp khác ledger loại tuần tự. Metadata roster trước lọc theo target.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S02_OPTIONS = {
    'decision_mode': None,
    'mode_evidence': None,
    'chronology_grade': None,
    'chronology_reason': None,
    'chronology_evidence': None
}
run_section("S02", S02_OPTIONS)

Cách đọc kết quả: Removal plot cần đọc tổng dòng theo bước, không cộng flags. Roster N_teams/placement consistency quyết định valid target. mode mapping có evidence, Other riêng. Không dùng test để ra quyết định EDA/model.

Hạn chế và bàn giao: Cleaned/metadata/split mới cùng split manifest; consumer S03. Timestamp có giây chưa đủ Grade A, tên người chơi không phải ID bất biến.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S03"></a>
## S03. Feature cơ sở và target

Câu hỏi kiểm tra: Normalized placement y=1-(rank-1)/(N_teams-1), winner1/last0. Cần roster đáng tin và N_teams>1; rank ngoài miền không clip. Survival valid và placement valid tách task.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Các biến raw/derived theo nhóm Combat/Movement/Support. Damage/kill, walk/total và assists/(kills+assists) mẫu số0 là missing, không epsilon. Metadata đội không tính lại trên model cohort.

Công thức cần kiểm chứng:

$$
D_{\mathrm{total}}=D_{\mathrm{walk}}+D_{\mathrm{ride}}
$$

$$
r_{\mathrm{walk}}=\frac{D_{\mathrm{walk}}}{D_{\mathrm{total}}}
$$

$$
r_{\mathrm{assist}}=\frac{\mathrm{assists}}{\mathrm{kills}+\mathrm{assists}}.
$$

$$
y_{\mathrm{placement}}=1-\frac{\mathrm{rank}-1}{N_{\mathrm{teams}}-1}.
$$

Ví dụ roster có 4 đội: rank 1 cho 1, rank 4 cho 0. Điều kiện mẫu số và roster áp dụng trước tính; mẫu số 0 là missing có lý do. Không clip để che lỗi.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S03_OPTIONS = {}
run_section("S03", S03_OPTIONS)

Bản 5.7: S03 dùng database tạm trên đĩa, JOIN metadata chỉ đọc cột cần thiết, kiểm khóa và bảo toàn số dòng. Validation tổng hợp mọi cột trong một lượt scan; công thức, kiểu và quy tắc missing không đổi.

Cách đọc kết quả: Dictionary/validation N/valid/missing/min/max giúp phát hiện sai phép chia. Ví dụ tính tay chỉ minh họa; đối chiếu với actual rows. Rate/velocity chứa survival không dùng S1 hoặc RQ1 primary-survival.

Hạn chế và bàn giao: Base Parquet giữ identity/lineage/targets/flags. Consumer S04; ghi đơn vị dựa nguồn, không gán verified từ dự án cũ.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S04"></a>
## S04. Combat Timing và hợp nhất

Câu hỏi kiểm tra: Proxy duration=max survival trong match, không official duration. Quy ước min60 giây, Early[0,1/3), Mid[1/3,2/3), Late[2/3,1]. Absolute timing hợp lệ giữ ngay cả phase invalid/out-of-range; không clip.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Event credits named non-self killer, valid match/time. Có thể gồm same-team/unknown-team, không enemy-only. First=min; avg=sum/time_count qua toàn shard, không trung bình chunk means.

Bản 5.7: ghi events/metadata/base một lượt vào 16 phần theo SHA256(match_id). Mọi dòng cùng trận ở cùng phần, dedup vẫn theo toàn nội dung gốc, không theo hash. Tính và JOIN từng phần trên đĩa, kiểm số dòng rồi mới ghép ra player_match.parquet. timing_partition_audit.csv và s04_execution.json ghi số dòng/cách chạy. Đổi lại cần thêm đĩa tạm và I/O, không hứa nhanh hơn. Chưa có resume từng phần hoặc tự tách phần lệch lớn; lỗi sẽ chạy lại S04, không đánh completed.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

Đơn vị: verified cần evidence xác minh. assumed_seconds chỉ được chấp nhận khi có evidence, approval=True và scope đúng dataset/version. Cấu hình bản 5.3 lưu chấp thuận ngày 18/10/2026 cho nguồn PUBG Version 3. Đây là giả định, không phải thông tin đã được tác giả nguồn xác nhận. Nếu S00 còn dòng SOURCE_TIME_UNIT_STATUS="pending" từ cấu hình cũ, cần đổi rõ thành "assumed_seconds" hoặc để trống để kế thừa config mới, không đổi thành verified.

Định nghĩa thời gian:

$$
T_m=\max_{i\in m}s_i,\qquad u_e=\frac{t_e}{T_m}
$$

$$
\overline{t}_{p,m}=\frac{\sum_e t_e}{n_{p,m}}.
$$

Qua các batch $b$, cộng $\sum_b \mathrm{sum}_b$ và $\sum_b n_b$ trước chia. Ví dụ một batch có 10, batch kia có 20/30/40: mean đúng 25, không phải mean của 10 và 30 là 20. First là global min 10.

Early $0\le u<1/3$, Mid $1/3\le u<2/3$, Late $2/3\le u\le1$. Với $T_m=300$, event 100 là Mid, 200 và 300 là Late. Đây là pha tương đối theo proxy hậu trận, không phải phút thực tế cố định.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S04_OPTIONS = {
    'partitions': None,  # Mặc định16; nếu đổi làm stale S03, chạy lại S03 rồi S04.
    'unit_status': None,
    'unit_evidence': None,
    'unit_assumption_approved': None,
    'unit_assumption_scope': None,
    'minimum_duration': None
}
run_section("S04", S04_OPTIONS)

Cách đọc kết quả: Đọc event audit/coverage/discrepancy trước hiểu feature. No-kill confirmed khác aggregate-kills>0 nhưng event missing; missing ratio không phải0. Phân bố phase có phase_valid denominator. LEFT JOIN không tăng số player rows.

Hạn chế và bàn giao: Player_match Parquet và full discrepancy Parquet mới; consumer S05-S09. Cần timing.unit_status/evidence từ nguồn/consistency mới; proxy chuẩn hóa tạo coupling với survival.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

Đơn vị: verified cần evidence xác minh. assumed_seconds chỉ được chấp nhận khi có evidence, approval=True và scope đúng dataset/version. Cấu hình bản 5.3 lưu chấp thuận ngày 18/10/2026 cho nguồn PUBG Version 3. Đây là giả định, không phải thông tin đã được tác giả nguồn xác nhận. Nếu S00 còn dòng SOURCE_TIME_UNIT_STATUS="pending" từ cấu hình cũ, cần đổi rõ thành "assumed_seconds" hoặc để trống để kế thừa config mới, không đổi thành verified.

<a id="S05"></a>
## S05. EDA bám thống kê và tiền xử lý C2

Câu hỏi kiểm tra: Giữ cực trị hợp lệ; boxplot/hist không tự tạo luật xóa. Log-transform hoặc giữ nguyên cần transform_reason. Mode comparisons dùng hành vi cho representation, không outcome chọn cụm.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Dùng train+validation cho biến đổi và thiết kế, không mở final test. Thống kê N/median/mean/std/quantiles/skew/missing/zero toàn scope; SQL cho dữ liệu lớn, không collect raw thành pandas.

Bản 5.8: một lượt quét lấy các thống kê có trạng thái nhỏ. Median/q25/q75 vẫn chính xác: sắp xếp từng cột toàn cục trên database đĩa, lấy vị trí cần thiết, dùng kernel nội suy Type-7 gốc trên tối đa hai đầu mút cho mỗi phân vị. Không dùng trung vị từng lô hoặc phân vị xấp xỉ. Histogram dùng lại min/max đã tính; giữ điều kiện valid_survival riêng cho target survival. s05_execution.json ghi cách thực thi, không phải bản lưu bền vững. Đổi lại cần đĩa tạm và đọc/ghi thêm.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S05_OPTIONS = {
    'decision_mode': None,
    'log_columns': None,
    'transform_reason': None
}
run_section("S05", S05_OPTIONS)

Cách đọc kết quả: Histograms có bins/units, missing khác zero. Nhìn lệch/phân tán/mode trước model. Dữ liệu người chơi lặp, outcome đội lặp và cơ hội tích lũy hành vi khi sống lâu đều hạn chế suy luận.

Hạn chế và bàn giao: EDA tables/plots/scope record; consumer S06/S07/S08. Nếu chưa chốt transforms lưu diagnostics rồi dừng; fixture không chốt thay dữ liệu thật.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S06"></a>
## S06. RQ1: quan hệ hành vi và outcome

Câu hỏi kiểm tra: Pair N phải có cho mỗi feature-target-mode. Constant/insufficient là undefined/NA, không0. Timing phase dùng duration proxy không primary với survival. P nhỏ khi N lớn không chứng minh quan hệ mạnh.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Pearson đo tuyến tính, Spearman đo quan hệ đơn điệu theo ranks; ranks/ties trên valid pairs toàn scope. Overall và từng mode riêng, giữ dấu hệ số.

Bản 5.8: mỗi feature-target-mode có bảng cặp hữu hạn riêng trên đĩa. Đếm tần số giá trị toàn cặp và cộng dồn để tạo average ranks, rồi JOIN hạng x/y theo từng bước, kiểm bảo toàn N. Với giá trị v, hạng trung bình là $r(v)=N_{<v}+(N_v+1)/2$. Không tái dùng rank của cặp khác hoặc trung bình tương quan từng phần. s06_execution.json ghi phương pháp; constant/empty vẫn undefined, không thay bằng0.

Hai hệ số:

$$
r=\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}
{\sqrt{\sum_i(x_i-\bar x)^2\sum_i(y_i-\bar y)^2}},
\qquad \rho=\mathrm{corr}(\mathrm{rank}(x),\mathrm{rank}(y)).
$$

Ties dùng average ranks trên đúng tập valid pairs. Mỗi cặp có N riêng; constant hoặc ít quan sát không có hệ số đáng diễn giải. Không tính riêng từng batch rồi trung bình hệ số.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S06_OPTIONS = {}
run_section("S06", S06_OPTIONS)

Cách đọc kết quả: Signed bars so Pearson/Spearman; xem khác mode và valid flags. Không kết luận walking gây survival hoặc early kills gây thắng. Mỗi phát biểu cần feature/target/scope/N/hệ số/giới hạn.

Hạn chế và bàn giao: rq1_relationships.csv và hình nguồn; consumer S10/S11. EDA quan hệ lựa chọn chưa dùng test; full description không được quay lại tune.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

<a id="S07"></a>
## S07. RQ2: KMeans riêng theo mode, bám C7

Câu hỏi kiểm tra: Chốt minimum_games từ retention/coverage, K từ inertia/silhouette/sizes/stability qua seed trên cùng reference profiles. Imputation/scaling học development; full descriptive fit chỉ sau khóa. Outcomes không input hoặc tiêu chí chọn K.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Player-mode profile Design3: mean/std combat, movement/support và timing ratios. Conditional means phải có valid counts; mean ratio khác pooled ratio. Singleton std không phải0. Games_played filter reliability, không predictor core.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

StandardScaler và mục tiêu KMeans:

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}
$$

$$
\min_{\{C_k,\mu_k\}}\sum_{k=1}^{K}\sum_{i\in C_k}\|z_i-\mu_k\|_2^2.
$$

Chuẩn hóa tránh khoảng cách bị biến có thang đo lớn chi phối; KMeans ưu tiên cụm gần dạng cầu và nhạy ngoại lệ/khởi tạo. Inertia giảm theo K nên cần silhouette/kích thước/stability và diễn giải cùng nhau.

Mean phase ratio là $\frac{1}{n_{\mathrm{valid\ matches}}}\sum_m r_m$. Hai trận 1/1 và 0/3 cho mean 0.5; pooled ratio 1/4=0.25 là thống kê khác. Độ lệch chuẩn mẫu dùng $n-1$; singleton là NA, không zero variance.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S07_OPTIONS = {
    'decision_mode': None,
    'gpu_ids': None,  # Kế thừa [0] trên Colab; không đổi khi stage đang chạy.
    'minimum_games': None,
    'candidate_k': None,
    'clusters': None
}
run_section("S07", S07_OPTIONS)

Cách đọc kết quả: Retention/K plots nêu diagnostic sample n/seed/rule; full fit không lấy mẫu. Standardized centers heatmap/cluster sizes giúp đặt tên hành vi. C5 outcomes chỉ sau assignments khóa; cluster IDs không so trực tiếp giữa modes.

Hạn chế và bàn giao: Lưu imputer/scaler/features/model/assignments/centers và receipts mỗi mode. Reload labels phải khớp. Null hoặc stale receipt dừng; GPU routing không fallback CPU. Timing proxy là giới hạn representation.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

Bản5.9: tổng hợp profile trên database đĩa, giữ nhãn seed đầu thay vì toàn bộ các seed, đọc metadata từng mode và reload theo lô50000. Davies-Bouldin tính khoảng cách hiệu tọa độ trực tiếp để tránh cancellation, không đổi định nghĩa hoặc quy tắc chọn K. Collect profile và full KMeans vẫn cần RAM/VRAM; không fit theo từng phần.

<a id="S08"></a>
## S08. RQ3: baseline và hồi quy tuyến tính C6

Câu hỏi kiểm tra: Train-only preprocessing và fit. Validation chọn thiết kế; G4 cần approval/reason/hash trước final test. T0/T1 same rows/target/split/model/preprocessing policy; mỗi nhánh fit trên train đúng cột. Không dùng actual để tạo prediction.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: S1 hồi quy survival với features không chứa target/descendants; P2 placement bỏ survival trực tiếp, vẫn có timing proxy coupling. Train mean/median và Linear Regression; T0 bỏ timing, T1 reuse P2 khi signature/cohort khớp.

Tự động: dùng OPTIONS trong cell thực thi; None kế thừa config. Chế độ manual vẫn giữ cổng chờ quyết định. Quyết định tự động được lưu vào manifest trước khi dùng; không dùng test để chọn tham số.

Mô hình hồi quy:

$$
\hat y_i=\beta_0+\sum_{j=1}^{p}\beta_j z_{ij}
$$

$$
\hat\beta=\arg\min_{\beta}\sum_{i\in \mathrm{train}}(y_i-\hat y_i)^2.
$$

Baseline mean/median chỉ học $y_{\mathrm{train}}$. Hệ số không là tác động nhân quả, đặc biệt khi predictors tương quan.

$$
MAE=\frac{1}{n}\sum_i|y_i-\hat y_i|
$$

$$
RMSE=\sqrt{\frac{1}{n}\sum_i(y_i-\hat y_i)^2}
$$

$$
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.
$$

R² có thể âm; target hằng hoặc n<2 là undefined. S1 metric theo đơn vị thời gian đã xác minh, placement trên [0,1]. Không gọi R² là tỷ lệ dự đoán đúng.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S08_OPTIONS = {
    'decision_mode': None,
    'gpu_ids': None,  # Kế thừa [0] trên Colab; không đổi khi stage đang chạy.
    'selection_reason': None,
    'selection_approved': None,
    'validation_hash': None
}
run_section("S08", S08_OPTIONS)

Cách đọc kết quả: Metric MAE/RMSE theo units, R² có thể âm và không accuracy. Bảng validation khác bảng final test. Survival vs placement khác đơn vị, không xếp hạng bằng MAE chung. Reload pipeline và batch prediction xác minh trước bàn giao.

Hạn chế và bàn giao: Bảy fit core logic, model/predictions/feature order/backend/cohort metadata mới. RAM/VRAM budget vượt ghi resource_limited; không đổi OLS/CPU/sample ngầm. Prior test exposure unknown phải công bố.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

Bản5.9: X/y float64 dạng memorymap tách khỏi metadata chuỗi; worker nhận/trả đường dẫn, predict/reload/output theo lô50000. Giữ một Pipeline.fit toàn train, bảy thí nghiệm và G4 trước test. Mmap không biến OLS thành incremental; metrics còn đọc một prediction frame đầy đủ và được tính trong cổng RAM. D5 chưa hoàn tất.

<a id="S09"></a>
## S09. Timing contribution, sai số và giải thích

Câu hỏi kiểm tra: Micro là player-match; match-aware xử lý trọng số match, team-aware placement kiểm actual nhất quán trong đội. Error bins chốt từ train, low-N/coverage rõ. Coefficients theo scaler/đa cộng tuyến, không nhân quả.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Đọc saved predictions, exact one-to-one row-ID/target/split trước delta; không dùng giao nhỏ che missing rows. Delta=candidate-reference, MAE/RMSE âm tốt hơn, R² dương tốt hơn.

So sánh:

$$
\Delta M=M_{\mathrm{T1}}-M_{\mathrm{T0}}.
$$

Paired bootstrap lấy cùng tập match IDs có hoàn lại cho hai nhánh, giữ số lần mỗi match xuất hiện; tính lại metric từ contributions, không trung bình RMSE/R² từng match.

Match-aware weights $w_i=1/n_{\mathrm{match}(i)}$, MAE là $\sum_iw_i|e_i|/\sum_iw_i$; R² dùng weighted mean và weighted SST đúng, không trung bình R²/match. Team-aware placement trung bình prediction trong đội sau kiểm target nhất quán.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S09_OPTIONS = {
    'bootstrap_replicates': None,
    'bootstrap_seed': None
}
run_section("S09", S09_OPTIONS)

Cách đọc kết quả: Residual=actual-predicted. Quan sát mode/miền target nơi sai cao nhưng không tối ưu lại test. CI nếu làm resample match paired có multiplicity, tính lại metrics; thiếu CI không vẽ khoảng/significance.

Hạn chế và bàn giao: Timing comparison/errors/coefficients/uncertainty và source figures; consumer S10. CI không bao gồm mọi bất định model selection hoặc repeated-player dependence.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

B?n5.10: ki?m ??y ?? row IDs/identity/target tr??c gh?p T0/T1, metrics v? error slices tr?n database ??a; bootstrap gi? RNG/multiplicity c?p tr?n v?i vectors numeric c? guard. M?u v? gi? ??ng RandomState/seed v? ch? ??c ba c?t Parquet theo l?50000; workspace ch? s? O(N) v?n c? guard. Kh?ng l?y trung b?nh RMSE/R2 t?ng l?, kh?ng sample fit/metrics.

Mỗi kết nối database đĩa dùng spill riêng, tránh ghi đè file tạm khi hai kết nối cùng hoạt động; giới hạn SQL không đổi.

<a id="S10"></a>
## S10. Khóa và kiểm tra kết quả

Câu hỏi kiểm tra: Thiếu core/stale/corrupt chặn khóa. Optional not_requested khác completed; không fake model/metric/artifact0. Release snapshot mới bất biến, read-back/checksum trước công bố.

Input/provenance: stage dependencies được kiểm từ manifest/checksum; phạm vi lấy từ S00, không lấy latest theo tên file.

Phương pháp và lý do: Chọn core artifacts theo completed compatible stages và required matrix, không quét mọi CSV/model còn sót làm official. Source/split/cohort/code/config/backend/actual run IDs đều cần provenance.

In [ ]:
# None: kế thừa cấu hình; sửa giá trị trước khi chạy cell này.
S10_OPTIONS = {}
run_section("S10", S10_OPTIONS)

Cách đọc kết quả: Bảng trạng thái/manifest khác số file tồn tại. Development release chỉ kiểm logic, report_ready=false. Khi canonical output ghi đè, release đã khóa phải còn đọc được.

Hạn chế và bàn giao: final_results pointer và release/manifest cụ thể. S11 report_only không cần raw/current checkpoint; chép bản portable với source/dependencies đã chuẩn bị.

Nếu bảng/hình chưa sinh hoặc stage pending, chưa có kết luận nghiên cứu. Hãy đọc reason, sửa đúng config/source rồi chạy lại S00 và phần này.

B?n5.10: inventory bytes/hash v? c? JSON tr??c copy; snapshot staging c?ngfilesystem, checksum/gi?i m?/load l?i tr??c rename; pointer cu?i. L?i copy/validate gi? b?n c? v? d?n staging. L?i pointer sau rename c? th? c?n snapshot ??y ?? ch?a tr? t?i, kh?ng completed. report_only preview nrows25 v? kh?ng collect fullprediction.

<a id="S11"></a>
## S11. Đọc kết quả đã khóa

report_only bỏ qua cài môi trường, tải raw, prepare/copy project và S01-S10. Điền RELEASE_MANIFEST là release.json cụ thể trên Drive; môi trường/code phải chuẩn bị từ trước. Bảng/hình đọc từ snapshot đã kiểm, không lấy current outputs hay số fixture cũ làm kết quả PUBG. Notebook phát hành không nhúng output thực thi.

In [ ]:
if EXECUTION_MODE == "report_only":
    summary = display_release(RELEASE_MANIFEST)
elif END_STAGE == "S10":
    pointer = read_json(ctx.path("manifests/final_results.json"))
    RELEASE_MANIFEST = str(ctx.root/pointer["manifest"])
    summary = display_release(RELEASE_MANIFEST)
else:
    print("Chưa tới finalization; chỉ đọc diagnostics/stage đã chạy, chưa có final summary")

<a id="S12"></a>
## S12. Kết luận khoa học, giới hạn và bàn giao
Mỗi phát biểu trong báo cáo phải dẫn bảng/hình/run: trên population/scope/mode nào, N bao nhiêu, hệ số/metric/delta thực đo gì và giới hạn gì. Không viết trước rằng timing cải thiện hoặc cụm nào tốt nhất.

Giới hạn bắt buộc: hồi cứu sau trận, dữ liệu quan sát, tên người chơi không ID bất biến, proxy duration, mismatch event/aggregate, phase-survival coupling, các đồng đội cùng outcome, chronology/selection/test exposure và giới hạn tài nguyên. CI theo match chưa giải quyết mọi phụ thuộc repeated players.

Liên hệ môn học: C1 quy trình KDD; C2 thống kê/cleaning/scaling; C6 hồi quy/residual/đa cộng tuyến; C7 KMeans/K/stability; C4 nguyên tắc đánh giá. C3/C5 không ép vào target liên tục.

Bộ nộp: một notebook có output, code/config/requirements/README, tables/figures và manifest cụ thể; báo cáo/slides cùng release. Demo report_only, không train raw lớn trong giờ bảo vệ. Các thành viên luyện giải thích per_mode/scaling/K, split group match, leakage, baseline, MAE/RMSE/R² và T0/T1.

Bàn giao: ghi root/project_id/code-config version, scope, stage cuối đã completed, pending reason, artifact paths và checksum, writer và section tiếp theo. File có tên hoặc cell cuối xanh không chứng minh toàn quy trình hoàn tất.

In [ ]:
if EXECUTION_MODE != "report_only":
    display(pd.DataFrame([{"stage":s,"status":r.get("status"),"reason":r.get("reason","")} for s,r in ctx.manifest().items()]))
    print("Project:", ctx.root)
    print("Checkpoint:", ctx.manifest_path)
    print("Giữ cả src/config/data/outputs và notebook Colab; không chỉ tải một file prediction.")
    if not USE_GOOGLE_DRIVE:
        print("CẢNH BÁO: project chỉ ở runtime, có thể mất khi VM bị thu hồi.")
else:
    print("Chỉ đọc release:", RELEASE_MANIFEST)


### Kiểm đọc lại project Colab

Cell sau kiểm checksum/giải mã Parquet của handoff qua filesystem đang mount, có thể đọc nhiều GiB. Không gọi đây là đã lưu Kaggle Version hoặc xác nhận đồng bộ máy chủ Google. Project Drive là nơi ghi trực tiếp, không phải bản backup độc lập. Stage bị ngắt giữa fit không có checkpoint giữa fit; giữ writer lock và chỉ takeover khi chắc phiên cũ đã dừng. report_only không tạo handoff hoặc ghi lại project.

In [ ]:
if VERIFY_HANDOFF and EXECUTION_MODE != "report_only":
    display(pd.DataFrame([verify_colab_handoff(project_root, EXECUTION_MODE)]))
else:
    print("Không chạy kiểm handoff trong lượt này; không xác nhận cloud/Kaggle save.")
